# Notebook 06: Statistical Validation

Bootstrap confidence intervals, sensitivity analysis, and Monte Carlo noise propagation for isotherm parameters.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.optimize import curve_fit
from utils_isotherms import langmuir, sips, generate_synthetic_data, extract_descriptors

## Generate reference data

In [ ]:
# True Sips parameters
Qmax_true, KS_true, m_true = 120.0, 0.08, 0.7
Ce, q_exp, q_true = generate_synthetic_data(
    sips, (Qmax_true, KS_true, m_true),
    Ce_range=(0.1, 100), n_points=25, noise_level=0.05, seed=42
)

# Fit Sips to get baseline
popt, pcov = curve_fit(sips, Ce, q_exp, p0=[100, 0.08, 0.8],
                       bounds=([0, 0, 0.1], [500, 10, 2]), maxfev=10000)
print(f'Baseline fit: Qmax={popt[0]:.1f}, KS={popt[1]:.4f}, m={popt[2]:.3f}')

## Bootstrap confidence intervals

Resample residuals with replacement to estimate parameter distributions.

In [ ]:
n_bootstrap = 1000
rng = np.random.default_rng(123)

residuals = q_exp - sips(Ce, *popt)
boot_params = []

for _ in range(n_bootstrap):
    boot_residuals = rng.choice(residuals, size=len(residuals), replace=True)
    q_boot = sips(Ce, *popt) + boot_residuals
    q_boot = np.maximum(q_boot, 0.01)
    try:
        p_boot, _ = curve_fit(sips, Ce, q_boot, p0=popt,
                              bounds=([0, 0, 0.1], [500, 10, 2]),
                              maxfev=10000)
        boot_params.append(p_boot)
    except RuntimeError:
        continue

boot_params = np.array(boot_params)
print(f'Successful bootstrap fits: {len(boot_params)}/{n_bootstrap}')

# 95% confidence intervals
param_names = ['Qmax', 'KS', 'm']
for i, name in enumerate(param_names):
    lo, hi = np.percentile(boot_params[:, i], [2.5, 97.5])
    print(f'{name}: {popt[i]:.4f}  95% CI: [{lo:.4f}, {hi:.4f}]')

In [ ]:
# Visualize bootstrap distributions
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for i, (ax, name) in enumerate(zip(axes, param_names)):
    ax.hist(boot_params[:, i], bins=40, density=True, alpha=0.7, color='steelblue')
    ax.axvline(popt[i], color='red', linestyle='--', label='Best fit')
    lo, hi = np.percentile(boot_params[:, i], [2.5, 97.5])
    ax.axvline(lo, color='orange', linestyle=':', label='95% CI')
    ax.axvline(hi, color='orange', linestyle=':')
    ax.set_xlabel(name)
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)
plt.suptitle('Bootstrap Parameter Distributions (Sips model)')
plt.tight_layout()
plt.show()

## Sensitivity analysis: effect of noise level on fitted parameters

In [ ]:
noise_levels = [0.01, 0.02, 0.05, 0.10, 0.15, 0.20]
n_repeats = 100
sensitivity = {name: [] for name in param_names}
sensitivity_std = {name: [] for name in param_names}

for noise in noise_levels:
    param_samples = []
    for seed in range(n_repeats):
        _, q_noisy, _ = generate_synthetic_data(
            sips, (Qmax_true, KS_true, m_true),
            Ce_range=(0.1, 100), n_points=25,
            noise_level=noise, seed=seed
        )
        try:
            p, _ = curve_fit(sips, Ce, q_noisy, p0=[100, 0.08, 0.8],
                             bounds=([0, 0, 0.1], [500, 10, 2]), maxfev=10000)
            param_samples.append(p)
        except RuntimeError:
            continue
    param_samples = np.array(param_samples)
    for i, name in enumerate(param_names):
        sensitivity[name].append(np.mean(param_samples[:, i]))
        sensitivity_std[name].append(np.std(param_samples[:, i]))

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
true_vals = [Qmax_true, KS_true, m_true]
for i, (ax, name) in enumerate(zip(axes, param_names)):
    ax.errorbar(noise_levels, sensitivity[name], yerr=sensitivity_std[name],
                fmt='o-', capsize=4, color='steelblue')
    ax.axhline(true_vals[i], color='red', linestyle='--', label='True value')
    ax.set_xlabel('Noise level (fraction)')
    ax.set_ylabel(name)
    ax.legend(fontsize=8)
plt.suptitle('Sensitivity of Sips Parameters to Noise')
plt.tight_layout()
plt.show()

## Monte Carlo propagation to universal descriptors

Propagate bootstrap parameter uncertainty through to {E_ads, sigma_H}.

In [ ]:
Eads_samples = []
sigma_samples = []

for p in boot_params:
    desc = extract_descriptors('Sips',
                               {'Qmax': p[0], 'KS': p[1], 'm': p[2]},
                               C0=1.0, T=298.0)
    Eads_samples.append(desc['Eads'])
    sigma_samples.append(desc['sigma_H'])

Eads_samples = np.array(Eads_samples)
sigma_samples = np.array(sigma_samples)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].hist(Eads_samples, bins=40, density=True, alpha=0.7, color='coral')
lo, hi = np.percentile(Eads_samples, [2.5, 97.5])
axes[0].axvline(lo, color='k', linestyle=':')
axes[0].axvline(hi, color='k', linestyle=':')
axes[0].set_xlabel('E_ads (kJ/mol)')
axes[0].set_ylabel('Density')
axes[0].set_title(f'E_ads: {np.mean(Eads_samples):.2f} [{lo:.2f}, {hi:.2f}]')

axes[1].hist(sigma_samples, bins=40, density=True, alpha=0.7, color='mediumpurple')
lo, hi = np.percentile(sigma_samples, [2.5, 97.5])
axes[1].axvline(lo, color='k', linestyle=':')
axes[1].axvline(hi, color='k', linestyle=':')
axes[1].set_xlabel('sigma_H (kJ/mol)')
axes[1].set_ylabel('Density')
axes[1].set_title(f'sigma_H: {np.mean(sigma_samples):.2f} [{lo:.2f}, {hi:.2f}]')

plt.suptitle('Monte Carlo Propagation of Uncertainty to Universal Descriptors')
plt.tight_layout()
plt.show()